# JAILJUDGE baseline and controls

Reproduce the two reports from saved verdicts. Baseline (B) and neutral (N) cover all 470 responses. Each refusal condition covers the 200 human-safe responses. Full-set DFR@5 is unmeasured. The tables use repeat 0 after checking agreement.

Place jailjudge-baseline-results.zip and jailjudge-control-results.zip in the working directory. In Colab the notebook prompts for both archives.

In [ ]:
from pathlib import Path
import json
from zipfile import ZipFile
import pandas as pd

archives = ['jailjudge-baseline-results.zip', 'jailjudge-control-results.zip']
if any(not Path(name).exists() for name in archives):
    from google.colab import files
    files.upload()

frames = []
for name in archives:
    with ZipFile(name) as archive:
        for model in ['llama_guard', 'shieldgemma', 'ministral', 'gemma']:
            records = [json.loads(line) for line in archive.read(f'{model}/verdicts.jsonl').splitlines()]
            frame = pd.DataFrame(records).assign(model=model)
            if name == archives[0]:
                frame['condition'] = 'B'
            frames.append(frame)
df = pd.concat(frames, ignore_index=True)
key = ['model', 'source_id', 'condition', 'mapping']
assert len(df) == 23280 and not df.duplicated(key + ['repeat']).any()
pairs = df.pivot(index=key, columns='repeat', values='verdict')
assert set(pairs.columns) == {0, 1} and pairs[0].eq(pairs[1]).all()
assert df.verdict.isin(['SAFE', 'UNSAFE']).all()
rows = df.loc[df['repeat'].eq(0)].copy()
rows.groupby(['model', 'mapping', 'condition']).source_id.nunique().unstack()

In [ ]:
rows['false_positive'] = ~rows.harmful & rows.verdict.eq('UNSAFE')
rows['false_negative'] = rows.harmful & rows.verdict.eq('SAFE')
rows.groupby(['model', 'mapping', 'condition'])[['false_positive', 'false_negative']].sum().loc[
    lambda counts: counts.index.get_level_values('condition').isin(['B', 'N'])
]

In [ ]:
safe = rows.loc[~rows.harmful]
safe.groupby(['model', 'mapping', 'condition']).false_positive.sum().unstack()